# Results Analysis

In [ ]:
def default_params(): 
    return {
        'current_model' : 'M6',
        'input_path' : '/causal_analysis_results',
        'output_plots' : '/correlation_plots'
    }
params = default_params()

### Imports

In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import os

In [3]:
def create_folder(path):
    if not os.path.exists(path):
        os.makedirs(path)

### Dataset loading

In [4]:
def load_json_dataframes(directory):
    """
    Reads all .json files from the specified directory, parses each as a pandas DataFrame,
    and returns a dictionary with file names (without extension) as keys and DataFrames as values.
    """
    dataframes = {}
    for filename in os.listdir(directory):
        if filename.endswith('.json'):
            filepath = os.path.join(directory, filename)
            key = os.path.splitext(filename)[0]
            dataframes[key] = pd.read_json(filepath)
    return dataframes

In [5]:
causal_results_dict = load_json_dataframes(f"{params['input_path']}/{params['current_model']}")

In [6]:
causal_results_dict['T1'].sort_values(by=['estimated_effect'], ascending=True)

,method_name,pearson_corr,spearman_corr,kendall_corr,estimated_effect,refutation_placebo_permute,refutation_unobserved_confounder,refutation_subset,pii_type
1,backdoor.propensity_score_matching,-0.351981,-0.351981,-0.351981,-0.167683,"{'result': {'p_value': 0.97, 'is_statistically...","{'result': None, 'new_effect': [-0.1691573337,...","{'result': {'p_value': 0.76, 'is_statistically...",key
0,backdoor.propensity_score_matching,-0.216188,-0.216188,-0.216188,-0.110251,"{'result': {'p_value': 1.0, 'is_statistically_...","{'result': None, 'new_effect': [-0.09410266810...","{'result': {'p_value': 0.68, 'is_statistically...",ip_address
5,backdoor.propensity_score_matching,-0.185945,-0.185945,-0.185945,-0.032552,"{'result': {'p_value': 0.67, 'is_statistically...","{'result': None, 'new_effect': [-0.03460455930...","{'result': {'p_value': 0.72, 'is_statistically...",password
2,backdoor.propensity_score_matching,-0.078540,-0.078540,-0.078540,-0.007519,"{'result': {'p_value': 0.89, 'is_statistically...","{'result': None, 'new_effect': [-0.0280510541,...","{'result': {'p_value': 0.84, 'is_statistically...",name
4,backdoor.propensity_score_matching,-0.041023,-0.041023,-0.041023,-0.001718,"{'result': {'p_value': 0.93, 'is_statistically...","{'result': None, 'new_effect': [-0.0540289071,...","{'result': {'p_value': 0.96, 'is_statistically...",username
3,backdoor.propensity_score_matching,-0.028230,-0.028230,-0.028230,0.000000,"{'result': {'p_value': 0.9500000000000001, 'is...","{'result': None, 'new_effect': [-0.00745541290...","{'result': {'p_value': 0.79, 'is_statistically...",email


In [7]:
causal_results_dict['T1'].groupby(['pii_type']).describe()

pearson_corr                                                        \
                  count      mean std       min       25%       50%       75%   
pii_type                                                                        
email               1.0 -0.028230 NaN -0.028230 -0.028230 -0.028230 -0.028230   
ip_address          1.0 -0.216188 NaN -0.216188 -0.216188 -0.216188 -0.216188   
key                 1.0 -0.351981 NaN -0.351981 -0.351981 -0.351981 -0.351981   
name                1.0 -0.078540 NaN -0.078540 -0.078540 -0.078540 -0.078540   
password            1.0 -0.185945 NaN -0.185945 -0.185945 -0.185945 -0.185945   
username            1.0 -0.041023 NaN -0.041023 -0.041023 -0.041023 -0.041023   

                     spearman_corr            ... kendall_corr            \
                 max         count      mean  ...          75%       max   
pii_type                                      ...                          
email      -0.028230           1.0 -0.028230  ...    -0.028230 -0.028230   
ip_address -0.216188           1.0 -0.216188  ...    -0.216188 -0.216188   
key        -0.351981           1.0 -0.351981  ...    -0.351981 -0.351981   
name       -0.078540           1.0 -0.078540  ...    -0.078540 -0.078540   
password   -0.185945           1.0 -0.185945  ...    -0.185945 -0.185945   
username   -0.041023           1.0 -0.041023  ...    -0.041023 -0.041023   

           estimated_effect                                              \
                      count      mean std       min       25%       50%   
pii_type                                                                  
email                   1.0  0.000000 NaN  0.000000  0.000000  0.000000   
ip_address              1.0 -0.110251 NaN -0.110251 -0.110251 -0.110251   
key                     1.0 -0.167683 NaN -0.167683 -0.167683 -0.167683   
name                    1.0 -0.007519 NaN -0.007519 -0.007519 -0.007519   
password                1.0 -0.032552 NaN -0.032552 -0.032552 -0.032552   
username                1.0 -0.001718 NaN -0.001718 -0.001718 -0.001718   

                                
                 75%       max  
pii_type                        
email       0.000000  0.000000  
ip_address -0.110251 -0.110251  
key        -0.167683 -0.167683  
name       -0.007519 -0.007519  
password   -0.032552 -0.032552  
username   -0.001718 -0.001718  

[6 rows x 32 columns]

In [8]:
causal_results_dict['T1'].loc[2]['refutation_subset']

{'result': {'p_value': 0.84, 'is_statistically_significant': False},
 'new_effect': -0.0105642633}

### Plots

In [9]:
def plot_correlations(causal_df, treatment, correlation, method_name):
    # Transforming the DataFrame using melt
    causal_df = causal_df[causal_df['method_name'] == method_name]
    correlation_plot_df = causal_df.melt(id_vars=['pii_type'], value_vars=[correlation, 'estimated_effect'],  var_name='type', value_name='value')

    # Create the plot
    plt.figure(figsize=(10, 6))
    barplot = sns.barplot(
        palette='mako',
        data=correlation_plot_df,
        x="pii_type",
        y="value",
        hue="type",
        alpha=0.8, 
        errorbar=None)

    # Improve x-axis readability
    plt.xlabel("PII Type", fontsize=12)
    plt.ylabel("Value", fontsize=12)
    plt.title(f"{correlation} vs ATE. ASR, Model [{params['current_model']}]", fontsize=14)

    # Move the legend outside for clarity
    plt.legend(title="Metrics", bbox_to_anchor=(1.05, 1), loc='upper left')
    
    output_dir = f"{params['output_plots']}/{params['current_model']}/{method_name}"
    create_folder(output_dir)
    plt.savefig(f"{output_dir}/{treatment}_{correlation}.png", bbox_inches="tight", dpi=300)
    plt.close()



In [10]:
#method_names = ['backdoor.generalized_linear_model', 'backdoor.propensity_score_matching', 'backdoor.propensity_score_stratification', 'backdoor.propensity_score_weighting']
method_names = ['backdoor.propensity_score_matching']

for treatment, causal_df in causal_results_dict.items():
    for correlation in ['pearson_corr', 'spearman_corr', 'kendall_corr']:
        for method_name in method_names:
            plot_correlations(causal_df, treatment, correlation, method_name)

In [11]:
causal_results_dict['T1'].columns

Index(['method_name', 'pearson_corr', 'spearman_corr', 'kendall_corr',
       'estimated_effect', 'refutation_placebo_permute',
       'refutation_unobserved_confounder', 'refutation_subset', 'pii_type'],
      dtype='object')

#### Summary

In [ ]:
type_map = {'easy': 'T0', 'hard': 'T1', 'ambiguous': 'T2'}
treatment = type_map['ambiguous']

summary_df = causal_results_dict[treatment][causal_results_dict[treatment]['method_name']=='backdoor.propensity_score_matching'].copy()
summary_df = summary_df[['pii_type','pearson_corr','estimated_effect', 'refutation_placebo_permute']].sort_values(by='estimated_effect', ascending=False)
summary_df['pearson_corr'] = summary_df['pearson_corr'].round(3)
summary_df['estimated_effect'] = summary_df['estimated_effect'].round(3)
summary_df['refutation_placebo_permute'] = summary_df['refutation_placebo_permute'].map(lambda result: result['new_effect'])
summary_df['refutation_placebo_permute'] = summary_df['refutation_placebo_permute'].round(4)
summary_df.sort_values(by='estimated_effect', ascending=False)


,pii_type,pearson_corr,estimated_effect,refutation_placebo_permute
3,email,-0.028,0.000,0.0024
4,username,-0.041,-0.002,-0.0084
2,name,-0.079,-0.008,0.0001
5,password,-0.186,-0.033,0.0008
0,ip_address,-0.216,-0.110,0.0019
1,key,-0.352,-0.168,-0.0019
